
<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin: 20px; height: 55px">

# NLP II: `CountVectorizer`, `TfidfVectorizer`, and Modeling

---

<img src="https://snag.gy/uvESGH.jpg" alt="drawing" width="800"/>

$$
\begin{eqnarray*}
\textbf{Fun Fact:  } \text{Word Clouds} &\neq& \text{Data Science}
\end{eqnarray*}
$$

[If you want to generate a word cloud in the shape of something **for art only**, check here.](https://medium.com/hackernoon/what-real-fake-news-says-about-obamas-presidency-4bf42be71ff1)

## Learning Objectives
---

- Extract features from unstructured text by fitting and transforming with `CountVectorizer` and `TfidfVectorizer`
- Describe how count vectorizers and TF/IDF vectorizers work
- Understand the many tuning parameters available in your vectorizer
- Implement `CountVectorizer` and `TfidfVectorizer` in a spam classification model.
- Use `GridSearchCV` and `Pipeline` with `CountVectorizer`

In [ ]:
# imports

# import CountVectorizer and TFIDFVectorizer from feature_extraction.text.

# Reminder of the Data Science Process
1. Define problem.
2. Gather data.
3. Explore data.
    - Yes, we can still do EDA with text data!
    - We also have to pre-process our text data to prepare it for modeling.
4. Model with data.
5. Evaluate model.
6. Answer problem.

# Introduction to Text Feature Extraction

The models we've learned, like linear regression, logistic regression, and k-nearest neighbors, take in an `X` and a `y` variable.
- `X` is a matrix/dataframe of real numbers.
- `y` is a vector/series of real numbers.

Text data (also called natural language data) is not already organized as a matrix or vector of real numbers. We say that this data is **unstructured**.

> This lesson will focus on how to transform our unstructured text data into a numeric `X` matrix.

# Spam Classification Model

One common application of NLP is predicting "spam" vs. "ham," or "spam" vs. "not spam."

Can we predict real vs. promotional texts just based on what is written?

> This data set was taken from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/sms+spam+collection).

In [ ]:
# Read in data.
spam = pd.read_csv('data/SMSSpamCollection', sep='\t', names=['label', 'message'])

# Check out first five rows.
spam.head()

In [ ]:
spam.shape

**(THREAD)**: In the context of this problem, what is a "document"?

## Let's get our data
---

Convert ham/spam into binary labels:
- 0 for ham
- 1 for spam

In [ ]:
# Create label column

Let's set up our data for modeling:
- `X` will be the `message` column. 
    - **NOTE**: `CountVectorizer` requires a vector, so make sure you set `X` to be a `pandas` Series, **not** a DataFrame.
    - That is: We index a single column with `[]`, we are not multiselecting columns with `[[]]`!
- `y` will be the `label` column

In [ ]:
X.shape # Note 1-dim!

In [ ]:
# Check what we need to check in a classification problem.

In [ ]:
# Split the data into the training and testing sets.

# Pre-Processing

Let's review some of the pre-processing steps for text data:

- Remove special characters
- Tokenizing
- Lemmatizing/Stemming
- Stop word removal

`CountVectorizer` actually can do a lot of this for us! It is important to keep these steps in mind in case you want to change the default methods used for each of these.

## `CountVectorizer`
---

The easiest way for us to convert text data into a structured, numeric `X` dataframe is to use `CountVectorizer`.

- **Count**: Count up how many times a token is observed in a given document.
- **Vectorizer**: Create a column (also known as a vector) that stores those counts.

Let's familiarize ourselves with the parameters by peeping the [docs](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html).

In [ ]:
# Instantiate a CountVectorizer.

In [ ]:
# Fit the vectorizer on our corpus.

In [ ]:
# Transform the corpus.

In [ ]:
# How big is this??
import sys
size_sparse = sys.getsizeof(X_train_cv)
size_dense = sys.getsizeof(X_train_cv.toarray())

print(f"Data when sparse = {size_sparse} bytes")
print(f"Data when dense  = {size_dense} bytes")

In [ ]:
# Old sklearn = .get_feature_names()
# New sklearn = .get_feature_names_out()

In [ ]:
# Convert X_train into a DataFrame.
# We will not actually use this for modeling,
# just a neat tool for EDA

In [ ]:
# Notice all the "junk" columns in the beginning. We can get rid of these
# by tuning some parameters or editing the preprocessing argument with our own function

In [ ]:
# Plot top occuring words

In [ ]:
# Word counts per document

## Tuning via EDA

At this point, we could fit a model (like a logistic regression model or $k$-nearest neighbors model) using our transformed data. We'll get to this, but first let's keep messing around with our tuning parameters to see their effect.

Let's examine some of the different hyperparameters of `CountVectorizer`:
- `stop_words`
- `max_features`, `max_df`, `min_df`
- `ngram_range`

In [ ]:
# Make sure to test:
# - stop_words
# - ngram_range
# - min_df
# - max_features

# BONUS: Can get your make custom stopword list
# from nltk.corpus import stopwords
# sw_nltk = stopwords.words('english')
# sw = sw_nltk + ['lt', 'gt']

cvec = CountVectorizer(
    # Customize here!
)

X_train_cv = cvec.fit_transform(X_train)
X_test_cv = cvec.transform(X_test)

X_train_df = pd.DataFrame(
    X_train_cv.toarray(), 
    columns=cvec.get_feature_names()
)

X_train_df.sum().sort_values(ascending = False).head(10).plot(kind='barh');

# Modeling

---

We may want to test lots of different values of hyperparameters in our CountVectorizer.

**(THREAD):** Is CountVectorizer an estimator or a transformer?

**(THREAD):** Why do we need a pipeline to GridSearch over our CountVectorizer hyperparameters?

In [ ]:
# Redefine training and testing sets (if needed)

## Baseline accuracy

We need to calculate baseline accuracy in order to tell if our model is better than null model (predicting the plurality class).

## Naive What Now?
We're going to use the Naive Bayes algorithm for our model of choice today. It is swappable for any classification model out there (including logistic regression, of course). But Naive Bayes has been shown to work well for NLP problems such as these, and we're always interested in learning one more new thing. So we'll use it today!

If you want to know the specifics of the algorithm, check out the appendix to this lesson.

In [ ]:
# Let's set a pipeline up with two stages:
# 1. CountVectorizer (transformer)
# 2. Multinomial Naive Bayes (estimator)

## `GridSearchCV`
---

At this point, you could use your `pipeline` object as a model:

```python
# Estimate how your model will perform on unseen data
cross_val_score(pipe, X_train, y_train, cv=3).mean() 

# Fit your model
pipe.fit(X_train, y_train)

# Training score
pipe.score(X_train, y_train)

# Test score
pipe.score(X_test, y_test)
```

Since we want to tune over the `CountVectorizer`, we'll load our `pipeline` object into `GridSearchCV`.

In [ ]:
# How can I figure out what hyperparamters I can tune in my pipeline?

In [ ]:
# Search over the following values of hyperparameters:
# Maximum number of features fit: 2000, 3000, 4000, 5000
# Minimum number of documents needed to include token: 2, 3
# Maximum number of documents needed to include token: 90%, 95%
# Check (individual tokens) and also check (individual tokens and 2-grams).

# Instantiate GridSearchCV.


<details><summary>How many models are we fitting here?</summary>

- 4 max_features
- 2 min_df
- 2 max_df
- 2 ngram_range
- 5-fold CV
- 4 * 2 * 2 * 2 * 5 + 1 = 161 models
</details>

In [ ]:
%%time
# Fit GridSearch to training data.

In [ ]:
# Best Params

In [ ]:
# Score model on training set.

In [ ]:
# Score model on testing set.

In [ ]:
# Get predictions

In [ ]:
# View confusion matrix

**(THREAD):** Is accuracy the best score here?

**(THREAD):** What's worse, a false positive or a false negative?

- Precision: Out of all the messages that the model predicted would be spam, 98.8% were actually spam.
- Recall: Out of all the messages that were actually spam, the model predicted this outcome correctly 88.7% of the time.
- Specificity:  Out of all the messages that were actually legitimate, our model predicted this outcome correctly 99.5% of the time (Tip: This is the recall of the negative class).


Congratulations! We've used `CountVectorizer` to transform our text data into something we can pass into a model.

But what if we want to do something more than just count up the occurrence of each token?

## Term Frequency-Inverse Document Frequency (TF-IDF) Vectorizer

---

When modeling, which word do you think tends to be the most helpful?
- Words that are common across all documents.
- Words that are rare across all documents.
- Words that are rare across some documents, and common across some documents.

<details><summary>Answer:</summary>

- Words that are common in certain documents but rare in other documents tend to be more informative than words that are common in all documents or rare in all documents.
- Example: If we were examining poetry over time, the word "thine" might be common in some documents but rare in most documents. The word "thine" is probably pretty informative in this case.
</details>

TF-IDF is a score that tells us which words are important to one document, relative to all other documents. Words that occur often in one document but don't occur in many documents contain more predictive power.

Variations of the TF-IDF score are often used by search engines as a central tool in scoring and ranking a document's relevance given a user query.
- If you want to see how it can be calculated, check out [the Wikipedia page](https://en.wikipedia.org/wiki/Tf%E2%80%93idf) and [`sklearn`](https://scikit-learn.org/stable/modules/feature_extraction.html#tfidf-term-weighting) page.

<img src="assets/tfidfvectorizer.png" alt="drawing" width="750"/>

[Source](https://towardsdatascience.com/nlp-learning-series-part-2-conventional-methods-for-text-classification-40f2839dd061).

## Practice Using the `TfidfVectorizer`

---

`sklearn` provides a TF-IDF vectorizer that works similarly to the CountVectorizer.
- The arguments `stop_words`, `max_features`, `min_df`, `max_df`, and `ngram_range` also work here.

In [ ]:
# Redefine training and testing sets.

In [ ]:
# Convert X_train into a DataFrame.
# We will not actually use this for modeling,
# this is just to visualize what is happening

Let's visualize the top words.

In [ ]:
# plot top words according to tf-idf sum

### Modeling Using the `TfidfVectorizer`

Let's set up a pipeline using tf-idf and Multinomial Naive Bayes.

<details><summary>What's the problem with this?</summary>

- Technically, we are supposed to have positive integers to use Multinomial Naive Bayes. Tf-idf does not give us positive integers.
- However, it will still work. Even the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html#sklearn.naive_bayes.MultinomialNB) says "The multinomial distribution normally requires integer feature counts. However, in practice, fractional counts such as tf-idf may also work."
</details>

In [ ]:
# Let's set a pipeline up with two stages:
# 1. tf-idf vectorizer (transformer)
# 2. Multinomial Naive Bayes (estimator)

In [ ]:
# What params can I search over?

In [ ]:
# Search over the following values of hyperparameters:
# Maximum number of features fit: 2000, 3000, 4000, 5000
# No stop words and english stop words
# Check (individual tokens) and also check (individual tokens and 2-grams).

In [ ]:
# Instantiate GridSearchCV.

In [ ]:
%%time
# Fit GridSearch to training data.

In [ ]:
# Score model on training set.

In [ ]:
# Score model on testing set.

In [ ]:
# Best params?

In [ ]:
# Confusion matrix


- Precision: Out of all the messages that the model predicted would be spam, 98.8% were actually spam.
- Recall: Out of all the messages that were actually spam, the model predicted this outcome correctly 88.7% of the time.
- Specificity:  Out of all the messages that were actually legitimate, our model predicted this outcome correctly 99.8% of the time (Tip: This is the recall of the negative class).


# Appendix: The Naive Bayes Model

Naive Bayes relies on [Bayes theorem](https://www.mathsisfun.com/data/bayes-theorem.html), which is just a probability rule that's just a rearranging of the "And" rule for probabilities we've already learned. It sounds fancy, but it's just a quick formula:

If we know $P(B|A)$, Bayes theorem allows us to calculate the probability of $P(A|B)$ by relating the probability of $P(A|B)$ to $P(B|A)$. 

$$
\begin{eqnarray*}
\text{Bayes' Theorem: } P(A|B) &=& \frac{P(B|A)P(A)}{P(B)}
\end{eqnarray*}
$$

- Let $A$ be that a message is spam.
- Let $B$ represent the words used in the message.

$$
\begin{eqnarray*}
\text{Bayes' Theorem: } P(A|B) &=& \frac{P(B|A)P(A)}{P(B)} \\
\Rightarrow P(\text{message is spam}|\text{words in message}) &=& \frac{P(\text{words in message}|\text{message is spam})P(\text{message is spam})}{P(\text{words in message})}
\end{eqnarray*}
$$

We want to calculate the probability that a post is spam **given** the words that are in the message! Our model can learn this from the training data.

**Naïve Bayes** makes the assumption that all features are independent of one another (this is why it is called *naïve*).

<details><summary>Why is this assumption not realistic with our data?</summary>
    
Text data is never independent! Certain words can change the context of a sentence when used with other words. The way language works, we have words that are more or less likely to follow other words.
</details>

Despite this assumption not being realistic with NLP data, we still use Naïve Bayes pretty frequently.
- It's a very fast modeling algorithm (which is great especially when we have lots of features and/or lots of data!).
- It is often an excellent classifier, outperforming more complicated models.

There are three common types of Naive Bayes models: Bernoulli Naive Bayes, Multinomial Naive Bayes, and Gaussian Naive Bayes.
- How do we pick which of the three models to use? It depends on our $X$ variable.
    - [Bernoulli Naive Bayes](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.BernoulliNB.html#sklearn.naive_bayes.BernoulliNB): when we have 0/1 variables.
    - [Multinomial Naive Bayes](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html#sklearn.naive_bayes.MultinomialNB): when our variables are positive integers.
    - [Gaussian Naive Bayes](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html#sklearn.naive_bayes.GaussianNB): when our features are Normally distributed.